 # Model Evaluation Notebook

 **Goal**: Process all unseen images in `data/dataset_test/` across 5 models and save results to a Pandas DataFrame.

| Column | Description |
|---|---|
| `filename` | Original filename (e.g., `dill_0.jpg`) |
| `ground_truth_class` | Class extracted from filename (before first `_`) |
| `{model}_class_top1` | Predicted class for that model |
| `{model}_confidence_top1` | Confidence score (0–1) for that model |

In [2]:
# -----------------------------------------
# 1. Setup & Dependencies
#------------------------------------------

import os
import time
from pathlib import Path
from typing import List, Tuple
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms
# Configure environment paths
import sys
from pathlib import Path
# Set PYTHONPATH equivalent for imports in notebook
env_path = Path("backend/app/src")
if str(env_path) not in sys.path:
    sys.path.append(str(env_path))

from herbs_detection.model_registry import MODEL_REGISTRY, ENABLED_KEYS
from herbs_detection.timm_predictor import TimmPredictor

In [ ]:
#------------------------------------------
#2. Configuration
#------------------------------------------

# Define paths and model mappings

TEST_DIR = Path("data/dataset_test")
OUTPUT_CSV = Path("evaluation_results.csv")
MODEL_MAP = {
    "efficientnet_sklearn": model_sklearn,
    "pytorch_large": model_pytorch_large,
    "resnet18": load_model,
    "illness": model_illness,
}
# Add a generic GCS fallback model key for GCS-backed models
# This should be removed once you switch to the consolidated wandb model
MODEL_MAP["gcs_cache"] = load_model  # Assuming GCS uses same loader
print(f"Models configured: {list(MODEL_MAP.keys())}")

In [ ]:
#------------------------------------------
# 3. Image Loader & Preprocessing
#------------------------------------------

#Define image transform (adjust based on your model requirements)
#For example, you may have a `get_transform` function defined elsewhere
#Here, we assume standard normalization for ImageNet/EF/ResNet

IMAGE_SIZE = (224,)

# If models have different input sizes, adjust accordingly

TRANSFORM = transforms.Compose([
  transforms.Resize((IMAGE_SIZE[0], IMAGE_SIZE[1])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
#------------------------------------------
# 4. Model Evaluation Loop
#------------------------------------------


def run_single_model_evaluation(model_predict_fn, model_name: str) -> pd.DataFrame:
    """
    Evaluates a single model on all test images and returns predictions.
    """
    print(f"  - Evaluating {model_name}...")
    results_list = []
    start_time = time.time()
    # Ensure model is loaded
    if hasattr(model_predict_fn, 'model') and model_predict_fn.model is None:
        model_predict_fn.model, _, _ = load_model()  # Re-load if needed
        # If model was already loaded, you would handle it differently
        # For this notebook, we assume the model object is already available
    for image_file in sorted(TEST_DIR.glob("*.jpg")):
        image_path = str(image_file)
        try:
            # Ground truth from filename (split on first underscore)
            filestem = image_file.stem
            ground_truth_class = filestem.split('_')[0]
            # Load image
            img = Image.open(image_path).convert("RGB")
            img_tensor = TRANSFORM(img)
            # Predict (assuming model.predict_fn takes (tensor, label_map))
            # Adjust this based on your actual prediction signature
            pred_result = model_predict_fn.predict(img_tensor)
            if pred_result is None:
                pred_class = "unknown"
                pred_conf = 0.0
            else:
                pred_class, pred_conf = pred_result
            results_list.append({
                "filename": image_path,
                "ground_truth_class": ground_truth_class,
                f"{model_name}_class_top1": pred_class,
                f"{model_name}_confidence_top1": pred_conf
            })
        except Exception as e:
            print(f"  - Error processing {image_path}: {e}")
            results_list.append({
                "filename": image_path,
                "ground_truth_class": "error",
                f"{model_name}_class_top1": "error",
                f"{model_name}_confidence_top1": 0.0
            })
    elapsed = time.time() - start_time
    print(f"  - Completed in {elapsed:.2f}s for {len(results_list)} images")
    return pd.DataFrame(results_list)

In [ ]:
#------------------------------------------
# 5. Main Evaluation Function
#------------------------------------------


def evaluate_all_models() -> pd.DataFrame:
    """
    Evaluates all configured models and returns combined results.
    """
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"Test directory not found: {TEST_DIR}")
    # Ensure all images exist and are valid
    img_files = list(TEST_DIR.glob("*.jpg"))
    if not img_files:
        raise ValueError(f"No .jpg files found in {TEST_DIR}")
    print(f"Found {len(img_files)} images in {TEST_DIR}")
    all_results = []
    # Evaluate each model
    for model_name, model_fn in MODEL_MAP.items():
        print(f"\n>>> Processing {model_name.upper()}...")
        df = run_single_model_evaluation(model_fn, model_name)
        all_results.append(df)
    # Combine all results
    combined_df = pd.concat(all_results, axis=1)
    return combined_df


In [ ]:
#------------------------------------------
#  6. Run Evaluation and Save Results
#------------------------------------------

try:
    # Run evaluation
    results_df = evaluate_all_models()
    # Save to CSV
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n=== RESULTS SUMMARY ===")
    print(f"Total images processed: {len(results_df)}")
    print(f"Columns: {list(results_df.columns)}")
    print(f"Saved to: {OUTPUT_CSV}")
    print(f"\nFirst 5 rows:\n{results_df.head()}")
    # Optional: Add ground truth column if needed for accuracy analysis
    results_df.to_csv(OUTPUT_CSV, index=False)
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
#------------------------------------------
#7. (Optional) Analysis Visualization
#------------------------------------------

def plot_results(results_df: pd.DataFrame):
    """                                                                                                                                                                                        
    Simple accuracy and confusion analysis for each model                                                                                                                                      
    """                                                                                                                                                                                        
    # Calculate ground truth accuracy (where filestem matches pred class)                                                                                                                      
    # Note: You'll need to clean filenames of .jpg suffix and compare with GT                                                                                                                  
                                                                                                                                                                                               
    import matplotlib.pyplot as plt                                                                                                                                                            
                                                                                                                                                                                               
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))                                                                                                                                              
                                                                                                                                                                                             
    model_names = [name for name in results_df.columns if name != 'filename' and name != 'ground_truth_class']                                                                                 
                                                                                                                                                                                             
    for i, col in enumerate(model_names[:2]):  # Plot first 2 models                                                                                                                           
        # Extract class predictions (remove confidence suffix if present)                                                                                                                    
        col_clean = col.replace('_class_top1', '')                                                                                                                                             
        # Extract ground truth column                                                                                                                                                          
        gt_col = 'ground_truth_class'                                                                                                                                                          
                                                                                                                                                                                               
        # Compute accuracy: match GT with prediction                                                                                                                                           
        accuracy = (results_df[col_clean] == results_df[gt_col]).mean() * 100                                                                                                                  
        ax[i].bar([col_clean], [accuracy])                                                                                                                                                     
        ax[i].set_title(f'{col_clean} Accuracy: {accuracy:.1f}%')                                                                                                                              
        ax[i].set_ylim([0, 100])                                                                                                                                                               
        ax[i].grid(axis='y', axislabel='Accuracy %')                                                                                                                                           
                                                                                                                                                                                               
    plt.tight_layout()                                                                                                                                                                         
    plt.show()                                                                                                                                                                                 
                                                                                                                                                                                               
# Uncomment to run visualization                                                                                                                                                               
# plot_results(results_df)     